# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import pprint

# Define the Croissant schema URL for FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (using .to_json for safe dict-view access)
metadata = dataset.metadata.to_json()
print(f"Dataset: {metadata['name']}")
print(f"Description: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.
We use the Croissant metadata to list all record set `@id`s and their associated fields.

Below, we enumerate all record sets and their fields, referencing each by `@id`.

In [ ]:
# Examine all record sets and their fields by their `@id`
recordsets = dataset.metadata.record_sets

if not recordsets:
    print("No record sets declared in metadata. You may need to inspect DataFiles or distributions.")
else:
    for rs in recordsets:
        print(f"Record Set @id: {rs.id}")
        print(f"  Name: {rs.name}")
        print(f"  Description: {rs.description}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - @id: {field.id}, name: {field.name}, data_type: {field.data_type}")
        print("")
    print(f"Total RecordSets: {len(recordsets)}")

# For convenience, collect record set @ids
record_set_ids = [rs.id for rs in recordsets] if recordsets else []

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis.
We use the record set and field `@id`s from the overview above.

_If no record sets are found, we attempt to infer available data files or distributions._

In [ ]:
# If record sets are defined, extract them. Else, try to inspect distributions for tabular data.
import warnings

dataframes = {}

if record_set_ids:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {df.shape[0]} records with {df.shape[1]} columns from record set: {record_set_id}")
        else:
            warnings.warn(f"No records found for record set {record_set_id}.")
    if dataframes:
        # Show the first available DataFrame
        sample_record_set_id = record_set_ids[0]
        print(f"\nFields in '{sample_record_set_id}':")
        print(dataframes[sample_record_set_id].columns.tolist())
        display(dataframes[sample_record_set_id].head())
    else:
        print("Did not load any records into DataFrames from declared record sets.")
else:
    # Fallback: enumerate distributions
    distributions = getattr(dataset.metadata, 'distributions', [])
    print(f"No explicit record sets. Dataset distributions:")
    if distributions:
        for dist in distributions:
            print(json.dumps(dist.to_json(), indent=2))
    else:
        print("No distributions found in the metadata.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section demonstrates how to filter, normalize, or group by fields using the `@id` of each field.

_Please customize the field IDs per your dataset's schema. The below example is illustrative and will look for a numeric field in the first loaded DataFrame._

In [ ]:
from pandas.api.types import is_numeric_dtype

if dataframes:
    # Choose a sample dataframe and try a numeric field
    record_set_id = next(iter(dataframes.keys()))
    df = dataframes[record_set_id]
    # Try to automatically pick a numeric field
    numeric_field = None
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        print("No numeric field detected in the first record set. Please specify a column (by @id) for numeric analysis.")
    else:
        print(f"Using field '{numeric_field}' for numeric analysis (using field @id).")
        threshold = df[numeric_field].quantile(0.75)  # e.g., upper quartile as threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered to records with {numeric_field} > {threshold}")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Sample of normalized {numeric_field} (Z-score):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field
        group_field = None
        for col in df.columns:
            if col != numeric_field and not is_numeric_dtype(df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
else:
    print("No DataFrames loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

_Example: Histogram of the selected numeric field._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {record_set_id}")
    plt.xlabel(numeric_field)
    plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to:
* Load data and metadata from a FAIR^2 Croissant schema,
* Review record sets, fields, and explore column-level data,
* Apply basic filtering, normalization, and visualization using field `@id`s.

You can now perform further domain-specific exploration or modeling with this pipeline as your starting point.